## 🔗 Follow Me

<p align="center">
  <a href="https://github.com/Ramakm">
    <img src="https://img.shields.io/badge/GitHub-Follow-black?style=for-the-badge&logo=github" />
  </a>
  &nbsp;
  <a href="https://www.instagram.com/techwith.ram/">
    <img src="https://img.shields.io/badge/Instagram-Follow-E4405F?style=for-the-badge&logo=instagram&logoColor=white" />
  </a>
  &nbsp;
  <a href="https://x.com/techwith_ram">
    <img src="https://img.shields.io/badge/Twitter-Follow-1DA1F2?style=for-the-badge&logo=twitter&logoColor=white" />
  </a>
</p>

# 06 · Evaluation

> **Run on:** Google Colab (Runtime → Change runtime type → **T4 GPU**, free tier works). Sections marked *CPU-only* run anywhere.

We compare the **base** model vs the **fine-tuned** model on the held-out `data/test.jsonl` (never seen in training).

In [ ]:
!pip install -q unsloth trl peft accelerate bitsandbytes datasets evaluate rouge_score

In [ ]:
import json, torch, math
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

BASE_MODEL  = "unsloth/Llama-3.2-3B-Instruct"
ADAPTER_DIR = "llama32-3b-finetuned-lora"
N_TEST = 50

test = [json.loads(l)["messages"] for l in open("data/test.jsonl")][:N_TEST]
prompts = [m[:-1] for m in test]          # everything except the final assistant answer
refs    = [m[-1]["content"] for m in test]

## 1. Generate with a model

In [ ]:
def generate_all(model, tokenizer, prompts, max_new_tokens=300):
    FastLanguageModel.for_inference(model)
    outs = []
    for p in prompts:
        ids = tokenizer.apply_chat_template(p, add_generation_prompt=True, return_tensors="pt").to("cuda")
        with torch.no_grad():
            g = model.generate(input_ids=ids, max_new_tokens=max_new_tokens, do_sample=False, use_cache=True)
        outs.append(tokenizer.decode(g[0][ids.shape[1]:], skip_special_tokens=True).strip())
    return outs

# --- base model
model, tok = FastLanguageModel.from_pretrained(BASE_MODEL, max_seq_length=2048, load_in_4bit=True)
tok = get_chat_template(tok, chat_template="llama-3.2")
base_out = generate_all(model, tok, prompts)
del model; torch.cuda.empty_cache()

# --- fine-tuned model (adapter folder is loaded on top of its base automatically)
model, tok = FastLanguageModel.from_pretrained(ADAPTER_DIR, max_seq_length=2048, load_in_4bit=True)
tok = get_chat_template(tok, chat_template="llama-3.2")
ft_out = generate_all(model, tok, prompts)

## 2. Side-by-side (eyeball it — most important check)

In [ ]:
for i in range(3):
    print("="*90, f"\nPROMPT: {prompts[i][-1]['content'][:300]}")
    print("\n--- BASE ---\n", base_out[i][:500])
    print("\n--- FINE-TUNED ---\n", ft_out[i][:500])
    print("\n--- REFERENCE ---\n", refs[i][:500])

## 3. Automatic metrics
ROUGE measures overlap with the reference; it is a rough signal for open-ended text, so combine it with the manual review above.

In [ ]:
import evaluate
rouge = evaluate.load("rouge")
for name, out in [("base", base_out), ("fine-tuned", ft_out)]:
    r = rouge.compute(predictions=out, references=refs)
    print(f"{name:11s} ROUGE-1 {r['rouge1']:.3f}  ROUGE-L {r['rougeL']:.3f}  avg len {sum(map(len,out))/len(out):.0f} chars")

## 4. Perplexity on the test set (fine-tuned)

In [ ]:
FastLanguageModel.for_training(model)
model.eval()
tot_loss, tot_tok = 0.0, 0
for m in test[:30]:
    ids = tok.apply_chat_template(m, return_tensors="pt").to("cuda")
    with torch.no_grad():
        loss = model(input_ids=ids, labels=ids).loss.item()
    tot_loss += loss * ids.shape[1]; tot_tok += ids.shape[1]
print("perplexity:", round(math.exp(tot_loss / tot_tok), 2))

## 5. Checklist before calling it done
- [ ] Fine-tuned answers match the desired style better than base
- [ ] No regression: still answers general questions sensibly (catastrophic forgetting check)
- [ ] Model stops cleanly (emits `<|eot_id|>`, no endless rambling)
- [ ] Val loss didn't diverge from train loss
- [ ] Tested on prompts *outside* your dataset distribution

## 6. Deploy
```python
# GGUF for Ollama / LM Studio
model.save_pretrained_gguf("llama32-3b-gguf", tok, quantization_method="q4_k_m")
```
```bash
# Ollama
echo 'FROM ./llama32-3b-gguf/unsloth.Q4_K_M.gguf' > Modelfile
ollama create my-llama32 -f Modelfile && ollama run my-llama32
```

🎉 End of the project.